# T07: Fabric Lakehouse Quick Start

## What You'll Learn

Microsoft Fabric Lakehouses store data in two areas: **Files** (raw landing zone) and **Tables** (managed Delta Lake). In this notebook you will:

1. **Generate** a retail dataset at small scale
2. **Write** to Parquet files (simulating the Lakehouse Files section)
3. **Write** to Delta Lake format (simulating the Lakehouse Tables section)
4. **Inspect** the file structure that was created
5. **Read back** the data to confirm round-trip fidelity

## Prerequisites

- Python 3.9 or later
- `pip install sqllocks-spindle` (uncomment the install cell below if needed)
- `pip install deltalake` (for Delta Lake support)

## Time Estimate

**~5 minutes** from start to finish.

In [ ]:
# Uncomment the lines below if you're running in an environment
# where sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle
# %pip install deltalake

print("Installation cell ready. Uncomment the pip lines above if needed.")

## Generate Retail Data at Small Scale

### What we're about to do
We'll generate a complete retail dataset using the `small` scale. This gives us enough data to see realistic file sizes without waiting long.

### Why this matters
In a real Fabric Lakehouse workflow, you'd land raw files into the **Files** section and then promote curated data to **Tables** as Delta Lake. Spindle lets you simulate both paths locally before deploying to Fabric.

### What to expect
A summary showing every generated table and its row count.

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain

result = Spindle().generate(domain=RetailDomain(), scale="small", seed=42)

print(result.summary())

## Write to Parquet Files (Simulating Lakehouse Files)

### What we're about to do
We'll export the generated data as Parquet files into a directory structure that mirrors a Fabric Lakehouse **Files** landing zone: `Files/landing/retail/`.

### Why this matters
Parquet is the standard columnar format for analytics workloads. When data lands in the Files section of a Lakehouse, it's typically Parquet or CSV. Writing Parquet locally lets you test your ingestion pipelines before connecting to Fabric.

### What to expect
One Parquet file per table, written under `./lakehouse_demo/Files/landing/retail/`.

In [ ]:
files = result.to_parquet("./lakehouse_demo/Files/landing/retail/")

print(f"Wrote {len(files)} Parquet files.")
for f in files:
    print(f"  {f}")

## Inspect the File Structure

### What we're about to do
We'll walk the `./lakehouse_demo` directory tree and display every Parquet file along with its size in bytes.

### Why this matters
Understanding file sizes and layout helps you estimate storage costs, plan partition strategies, and verify that your data landed where you expected.

### What to expect
A listing of all `.parquet` files with their relative paths and sizes.

In [ ]:
from pathlib import Path

print("=== Lakehouse File Structure ===")
for f in sorted(Path("./lakehouse_demo").rglob("*.parquet")):
    print(f"  {f.relative_to('./lakehouse_demo')}: {f.stat().st_size:,} bytes")

## Write to Delta Lake Format

### What we're about to do
We'll export the same dataset to Delta Lake format, simulating the Lakehouse **Tables** section. Delta adds ACID transactions, time travel, and schema enforcement on top of Parquet.

### Why this matters
Fabric Lakehouse Tables are Delta Lake tables. By writing Delta locally, you can test downstream notebooks and pipelines that expect Delta semantics — things like `MERGE`, `UPDATE`, and time-travel queries.

### What to expect
A Delta Lake directory structure with `_delta_log/` transaction logs alongside Parquet data files.

In [ ]:
delta_files = result.to_delta("./lakehouse_demo/Tables/retail/")

print(f"Wrote {len(delta_files)} Delta tables.")
for f in delta_files:
    print(f"  {f}")

# Show the Delta log structure for the first table
delta_dirs = sorted(Path("./lakehouse_demo/Tables/retail/").iterdir())
if delta_dirs:
    first_table = delta_dirs[0]
    print(f"\n=== Delta structure for {first_table.name} ===")
    for item in sorted(first_table.rglob("*")):
        print(f"  {item.relative_to(first_table)}")

## Read Back the Data

### What we're about to do
We'll read the Parquet files back into DataFrames and compare row counts to the originals. This confirms that the write-read round trip preserves all data.

### Why this matters
Round-trip fidelity is essential. If you lose rows or corrupt data types during export, your downstream tests will be unreliable. This step proves the pipeline is lossless.

### What to expect
Matching row counts between the original in-memory data and the data read back from Parquet.

In [ ]:
import pandas as pd
from pathlib import Path

print("=== Round-Trip Verification ===")
parquet_dir = Path("./lakehouse_demo/Files/landing/retail/")
for pf in sorted(parquet_dir.glob("*.parquet")):
    table_name = pf.stem
    df_read = pd.read_parquet(pf)
    original_rows = len(result.tables[table_name])
    read_rows = len(df_read)
    status = "MATCH" if original_rows == read_rows else "MISMATCH"
    print(f"  {table_name}: original={original_rows}, read_back={read_rows} [{status}]")

print("\nRound trip complete!")

## What's Next?

You've simulated a complete Fabric Lakehouse landing — both Files (Parquet) and Tables (Delta Lake). Here's where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T08: Fabric SQL Database Quick Start** | Generate SQL INSERT scripts for Fabric SQL Database |
| **T09: Streaming Events** | Stream real-time events with configurable throughput and anomalies |
| **T11: Chaos Engineering** | Inject schema drift, null corruption, and referential breakage |

Happy generating!